# Contoso Retail — Sales Cleaning (Bronze → Silver)

**Case study:** Contoso Retail wants trustworthy sales analytics. Raw point-of-sale exports land in the Lakehouse `Files` area (bronze, as-is from the source system) with the mess you'd expect from a real retailer: mixed date formats, string-typed prices, duplicate transactions, and a few orphaned foreign keys.

This notebook is the PySpark port of `fabric/01-fabric-fundamentals/scripts/clean_sales_data_local.py`. Same cleaning rules, but running as a real Spark job against a Fabric Lakehouse over **OneLake** instead of pandas on a laptop.

**Medallion stage:** Bronze (`Files/raw/*.json`) → Silver (`Tables/silver_sales`).

Before running: attach this notebook to your Lakehouse (Lakehouse explorer on the left → **Add Lakehouse**), and upload `sales_raw.json`, `products.json`, `customers.json` into `Files/raw/` via the Lakehouse UI.

## 1. Read bronze JSON from OneLake

`spark.read.json` on a Fabric notebook reads straight from OneLake via the Lakehouse's relative `Files/` path — no connection string, no mounting. That's the whole point of OneLake: one logical copy of the data, addressable the same way from every engine.

In [ ]:
sales_raw = spark.read.option("multiline", True).json("Files/raw/sales_raw.json")
customers_raw = spark.read.option("multiline", True).json("Files/raw/customers.json")
products_raw = spark.read.option("multiline", True).json("Files/raw/products.json")

print(f"Raw sales rows: {sales_raw.count()}")
sales_raw.printSchema()
display(sales_raw.limit(10))

## 2. Deduplicate and normalize strings

Same rule as the local script: drop exact-duplicate rows, trim whitespace on ID columns, and standardize `payment_method` casing so `"credit card"` and `"Credit Card"` aggregate together downstream.

In [ ]:
from pyspark.sql import functions as F

sales = sales_raw.dropDuplicates()

for col in ["sale_id", "customer_id", "product_id", "store_id"]:
    sales = sales.withColumn(col, F.trim(F.col(col)))

sales = sales.withColumn(
    "payment_method", F.initcap(F.trim(F.col("payment_method")))
)

print(f"Rows after de-dup: {sales.count()}")

## 3. Cast numeric types

`unit_price` arrives as either a float or a quoted string (`"19.99"`). `cast("double")` coerces both; a value that truly isn't numeric becomes `null`, which we catch in the next step rather than let it silently become `0`.

In [ ]:
sales = (
    sales.withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("discount_pct", F.col("discount_pct").cast("int"))
)

## 4. Parse mixed-format dates

The raw export has four different date shapes (ISO, ISO with time, `MM/DD/YYYY`, `DD-Mon-YYYY`). `to_date` only parses one format per call, so we try each candidate format and `coalesce` the first one that worked.

In [ ]:
date_candidates = [
    F.to_date(F.col("sale_date"), "yyyy-MM-dd"),
    F.to_date(F.col("sale_date"), "yyyy-MM-dd'T'HH:mm:ss"),
    F.to_date(F.col("sale_date"), "MM/dd/yyyy"),
    F.to_date(F.col("sale_date"), "dd-MMM-yyyy"),
]

sales = sales.withColumn("sale_date", F.coalesce(*date_candidates))

## 5 & 6. Drop rows that fail required-field and business-rule checks

- Missing/blank `customer_id`, `unit_price`, or `sale_date` → can't reconcile, drop.
- Non-positive `quantity` or `unit_price` → bad data, drop.
- `discount_pct` outside 0–100 → bad data, drop.

In [ ]:
sales = sales.withColumn(
    "customer_id",
    F.when(F.col("customer_id") == "", None).otherwise(F.col("customer_id")),
)

sales_clean = sales.filter(
    F.col("customer_id").isNotNull()
    & F.col("unit_price").isNotNull()
    & F.col("sale_date").isNotNull()
    & (F.col("quantity") > 0)
    & (F.col("unit_price") > 0)
    & (F.col("discount_pct") >= 0)
    & (F.col("discount_pct") <= 100)
)

print(f"Rows after validity filters: {sales_clean.count()}")

## 7. Flag (don't drop) orphan foreign keys

A `customer_id` or `product_id` with no matching dimension row might be a late-arriving dimension, not necessarily bad data — so we flag it for downstream triage instead of silently dropping the sale.

In [ ]:
known_customers = customers_raw.select("customer_id").distinct()
known_products = products_raw.select("product_id").distinct()

sales_clean = (
    sales_clean.join(
        known_customers.withColumnRenamed("customer_id", "_c"),
        sales_clean.customer_id == F.col("_c"),
        "left",
    )
    .withColumn("is_orphan_customer", F.col("_c").isNull())
    .drop("_c")
    .join(
        known_products.withColumnRenamed("product_id", "_p"),
        sales_clean.product_id == F.col("_p"),
        "left",
    )
    .withColumn("is_orphan_product", F.col("_p").isNull())
    .drop("_p")
)

## 8. Recompute line_total and write to the silver table

`saveAsTable` writes a managed Delta table into the Lakehouse's `Tables/` area — this is what shows up in the Lakehouse explorer and is queryable from SQL endpoint, Power BI, or another notebook, all backed by the same OneLake files.

In [ ]:
sales_clean = sales_clean.withColumn(
    "line_total",
    F.round(
        F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_pct") / 100), 2
    ),
)

display(sales_clean.limit(10))

sales_clean.write.format("delta").mode("overwrite").saveAsTable("silver_sales")

print("Wrote silver_sales table.")
print(f"Orphan customers: {sales_clean.filter('is_orphan_customer').count()}")
print(f"Orphan products:  {sales_clean.filter('is_orphan_product').count()}")